*Statistička analiza podataka o knjigama (Goodreads dataset)*

Ovaj dataset sadrži informacije o raznim ljubavnim romanima i klasicima
Tri zavisna podatka koja cu koristiti su: book lenght, rating i number of ratings

*LINERNA REGRESIJA*

Formula: Y=Beta0 + beta1x + x (y zavisna, x nezavisna promenljiva, beta0 je konstanta a beta1 koef regresije). Cilj regresije je da se ispita uticaj nezavisnih promenljivih na zavisne, i predvidjanje njihovih vrednosti. Ovde zelimo da saznamo da li duže knjige imaju više ili niže prosečne ocene. Linearni model pokušava da pronađe najbolju pravu liniju  koja prolazi kroz skup tačaka: y= x1+x2 puta nezavisna promenljiva

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm
from scipy import stats

df=pd.read_csv('books.csv')
print(df.head())

df.plot('book length', 'rating', style='o')
plt.xlabel('Book Length')
plt.ylabel('Rating')
plt.title('Zavisnost prosečne ocene od dužine knjige')

Rez1 = sm.ols('rating ~ Q("book length")', data=df).fit()
Rez2 = sm.ols('rating ~ Q("book length")', data=df[:-1]).fit()

plt.plot(df.loc[Rez1.fittedvalues.index, 'book length'], Rez1.fittedvalues, 'r', label='Svi podaci')
plt.plot(df.loc[Rez2.fittedvalues.index, 'book length'], Rez2.fittedvalues, 'g', label='Bez outliera')


plt.scatter(df['book length'], df['rating'], color='blue', label='Podaci')

plt.xlabel('Book Length')
plt.ylabel('Rating')
plt.title('Zavisnost prosečne ocene od dužine knjige')
plt.legend()
plt.show()

print(Rez1.summary())
print(Rez2.summary())

Ovde smo napravili zavisnost prosecne ocene od duzine knjige: Crtali smo dijagram rasipanja. Kod Rez1 su svi podaci a kod Rez 2 smo oduzeli poslednju tacku
Ovde smo napravili statisticku analizu a dalje je u nastavku.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

df = pd.read_csv('books.csv')
print(df.columns)
df = df.dropna(subset=['book length', 'rating'])

X = df['book length'].values.reshape(-1, 1)
Y = df['rating'].values.reshape(-1, 1)

org = LinearRegression()
cln = LinearRegression()

org.fit(X, Y)
cln.fit(X[:-1], Y[:-1])

print("R^2 sa svim podacima:", org.score(X, Y))
print("R^2 bez outliera:", cln.score(X[:-1], Y[:-1]))

plt.plot(df['book length'][:-1], df['rating'][:-1], 'bo', label='Podaci')
plt.plot(df['book length'][-1:], df['rating'][-1:], 'r*', ms=20, lw=10, label='Outlier')

plt.xlabel('Book Length')
plt.ylabel('Rating')
plt.title('Linearna regresija dužine knjige i prosečne ocene')

test = np.c_[np.arange(df['book length'].min(), df['book length'].max(), 10)]

plt.plot(test, org.predict(test), 'k--', label='Svi podaci')
plt.plot(test, cln.predict(test), 'k', label='Bez outliera')

plt.legend()
plt.show()

Ovo je masinski (praktican deo) pristup linernoj regresiji fokusiran više na predikciju i ocenu modela.
X (nezavisna promenljiva) i Y (zavisna promenljiva), kreiraju se dva modela: jedan sa svim podacima, drugi bez outliera.
Podaci se prikazuju i x/y osa se podesavaju po potrebi. Generisu se nove tačake za predikciju i linije plt.plot prikazuju linije predikcije

*VIŠESTRUKA REGRESIJA*

Cilj višestruke regresije u ovom projektu je da se ispita kako više faktora istovremeno utiče na prosečnu ocenu knjige, višestruka regresija omogućava da uključimo više nezavisnih promenljivih

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm
from scipy import stats
from statsmodels.formula.api import ols

df = pd.read_csv('books.csv')
print(df.columns)

model1 = ols('rating ~ Q("book length")', data=df).fit()
print("\nModel 1: Zavisnost ocene od dužine knjige")
print(model1.summary())

model2 = ols('rating ~ Q("book length") + Q("number of ratings")', data=df).fit()
print("\nModel 2: Zavisnost ocene od dužine knjige i broja ocena")
print(model2.summary())

model3 = ols('rating ~ Q("book length") * Q("number of ratings")', data=df).fit()
print("\nModel 3: Interakcija između dužine knjige i broja ocena")
print(model3.summary())

Kao zavisnu promenljivu koristimo rating a kao nezavisne book length i number of ratings. U modelu1 pokazujemo zavisnost prosecne ocene samo od duzine knjige. U modelu2 zavisnost prosečne ocene od dužine knjige i broja ocena. U modelu3 uključujemo i interakciju između dužine knjige i broja ocena. Summary nam daje značajnost svake promenljive i ukupnu preciznost modela.

*POLIMORSKA REGRESIJA*

Cilj polinomne regresije u ovom projektu je da se ispita da li postoji nelinearna veza između dužine knjige i prosečne ocene. Za razliku od linearne regresije polimorska omogućava da se uoče zakrivljene linije i tačke preokreta

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

df = pd.read_csv('books.csv')
df = df.dropna(subset=['book length', 'rating'])
X = df['book length'].values.reshape(-1, 1)
y = df['rating'].values.reshape(-1, 1)

lin = LinearRegression()
lin.fit(X, y)

poly = PolynomialFeatures(degree=4)
X_poly = poly.fit_transform(X)
lin_poly = LinearRegression()
lin_poly.fit(X_poly, y)
plt.scatter(X, y, color='blue')

plt.plot(X, lin.predict(X), color='red', label='Linear Regression')
plt.title('Linear vs Polynomial Regression')
plt.xlabel('Book Length')
plt.ylabel('Rating')
plt.legend()
plt.show()
plt.scatter(X, y, color='blue')

X_sorted = np.sort(X, axis=0)
plt.plot(X_sorted, lin_poly.predict(poly.fit_transform(X_sorted)),
         color='green', label='Polynomial Regression (degree=4)')
plt.title('Polynomial Regression - Rating vs Book Length')
plt.xlabel('Book Length')
plt.ylabel('Rating')
plt.legend()
plt.show()
pred_length = 500
pred_array = np.array([[pred_length]])

print(f"Predikcija linearne regresije za {pred_length} strana:",
      lin.predict(pred_array)[0][0])
print(f"Predikcija polinomske regresije za {pred_length} strana:",
      lin_poly.predict(poly.fit_transform(pred_array))[0][0])

U ovoj regresiji i dalje gledamo kako se ocena knjige menja u zavisnosti od duzine knjige ali sada umesto prave linije imamo krivu, zato se ova regresija zove i nelinerna.
Pred predstavlja predikciju za odredjenu vrednost (npr. knjiga od 500 strana)

*ESTIMACIJA USLOVNIH I BEZUSLOVNIH VEROVATNOĆA ODREĐENIH DOGAĐAJA*

Bezuslovna verovatnoca, formula: P(A)=m/n gde je m povoljan br ishoda a n ukupan br ishoda. Kod uslovne verovatnoce: P(A│B)=P(AB)/P(B). P(A│B) znaci A pod uslovom B, P(AB) je istovremeno desavanje A i B, dok je P(B) verovatnoca desavanja B

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("books.csv")
atributi = ["rating", "book length"]
df = df[atributi]
df.head()

bins = [0, 200, 400, 600, float('inf')]
labels = ["< 200 str.", "200–400 str.", "400–600 str.", "> 600 str."]
df["length_category"] = pd.cut(df["book length"], bins=bins, labels=labels)

data_temp = df["length_category"].value_counts().sort_index()
P_duzina = pd.DataFrame((data_temp / data_temp.sum()))
P_duzina.columns = ["Verovatnoća"]
P_duzina.columns.name = "Kategorija dužine knjige"

P_duzina.plot.bar(figsize=(10,6), fontsize=14, color="skyblue")
plt.ylabel("Verovatnoća", fontsize=14)
plt.xlabel("Dužina knjige", fontsize=14)
plt.title("Bezuslovna verovatnoća dužine knjige", fontsize=16)
plt.show()

granica = 4.0  
data_temp = (df["rating"] >= granica).value_counts()
P_visoka = pd.DataFrame(data_temp / data_temp.sum())
P_visoka.index = ["Niska ocena", "Visoka ocena"]
P_visoka.columns = ["Verovatnoća"]
P_visoka.columns.name = "Ocena"

print(P_visoka)

P_visoka.plot.bar(figsize=(8,5), fontsize=14, color=["lightcoral", "mediumseagreen"])
plt.xlabel("Ocena", fontsize=14)
plt.ylabel("Verovatnoća", fontsize=14)
plt.title("Bezuslovna verovatnoća visoke ocene", fontsize=16)
plt.show()
data_temp = df.loc[df["rating"] >= granica, "length_category"]

P_duzina_pod_visoka = pd.DataFrame((data_temp.value_counts() / data_temp.shape[0]).sort_index())
P_duzina_pod_visoka.columns = ["Verovatnoća"]
P_duzina_pod_visoka.columns.name = "Kategorija dužine knjige"

print("Verovatnoća dužine knjige pod uslovom da ima visoku ocenu:")
P_duzina_pod_visoka.plot(kind='bar', figsize=(10,6), fontsize=14, color="goldenrod")
plt.xlabel("Dužina knjige", fontsize=14)
plt.ylabel("Verovatnoća", fontsize=14)
plt.title("Uslovna verovatnoća dužine knjige ako ima visoku ocenu", fontsize=16)
plt.show()

P_visoka_pod_duzina = P_duzina_pod_visoka * P_visoka.loc["Visoka ocena"] / P_duzina
print("Verovatnoća visoke ocene pod uslovom dužine knjige:")
P_visoka_pod_duzina.plot.bar(figsize=(10,6), fontsize=14, color="orchid")
plt.xlabel("Dužina knjige", fontsize=14)
plt.ylabel("Verovatnoća", fontsize=14)
plt.title("Verovatnoća visoke ocene pod uslovom dužine knjige", fontsize=16)
plt.show()

Ovde smo uzeli book length i rating. Računali smo za bezuslovnu verovatnocu da knjiga ima viskou ocenu a za uslovnu da knjiga ima visoku ocenu AKO je duža od određenog broja stranica.
Knjige se svrstaju u četiri grupe po dužini: kratke, srednje, duže i vrlo duge i racuna se verovatnoća da knjiga ima visoku ocenu (≥ 4), zatim se poredi kako duzina utice na tu verovatnocu. U grafovima se vidi gde su "najbolje" knjige. Npr kod knjiga sa vecim brojem str (400) mozemo videti veci udeo ocene, obimnost=kvalitet

P_duzina_pod_visoka (Kolika je šansa za visoku ocenu ako znamo duzinu knjige) i P_visoka_pod_duzina (Koliko je visoka cena po duzini) su USLOVNE VEROVATNOĆE; 
P_duzina (Kolika je duzina knjige) i P_visoka (Koliko ima visokih ocena) su BEZUSLOVNE VEROVATNOĆE

*ESTIMACIJA PARAMETARA ZDRUŽENIH RASPODELA (KOVARIJANSA I KOEFICIJENT KORELACIJE)*

Kovarijansa je osnovna mera koja pokazuje zajednicku promenu dve promenljive. Formula: cov(x,y)=1/n puta sum(xi-x)puta(yi-y). Zatim mozemo dobiti koef korelacije koji meri stepen povezanosti izmedju dve promenjive: r=cov(x,y)/(sx)puta(sy),  3 rezultata-->-1(savrseno negativna linearna veza), 0(nema linerne veze), 1(savrseno pozitivna linearna veza)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("books.csv")

X = df["book length"]
Y = df["rating"]

print("Prvih par redova:")
print(df[["book length", "rating"]].head())

cov_matrix = np.cov(X, Y)
cov_value = cov_matrix[0, 1]
print(f"\nKovarijansa između dužine knjige i prosečne ocene: {cov_value:.4f}")

corr_value = np.corrcoef(X, Y)[0, 1]
print(f"Koeficijent korelacije (r): {corr_value:.4f}")

if corr_value > 0.5:
    print("Postoji jaka pozitivna korelacija — duže knjige generalno imaju veće ocene.")
elif corr_value > 0.2:
    print("Postoji slaba pozitivna korelacija — duže knjige blago imaju veće ocene.")
elif corr_value < -0.5:
    print("Jaka negativna korelacija — duže knjige imaju niže ocene.")
elif corr_value < -0.2:
    print("Slaba negativna korelacija — duže knjige blago imaju niže ocene.")
else:
    print("Nema značajne korelacije između dužine knjige i prosečne ocene.")

plt.figure(figsize=(8,6))
plt.scatter(X, Y, alpha=0.6)
plt.title("Zavisnost prosečne ocene od dužine knjige", fontsize=15)
plt.xlabel("Dužina knjige (stranice)", fontsize=12)
plt.ylabel("Prosečna ocena", fontsize=12)
plt.grid(True)
plt.show()

U ovoj metodi estimacije gledamo da dužina knjige utiče na njenu ocenu. Prvo računamo kovarijansu a zatim koeficijent korelacije.
U rezultatima pokazujemo da li je korelacija jaka ili slaba, pozitivna ili negativnai to sve prikazujemo grafikom

*ESTIMACIJA INTERVALA POVERENJA ODREĐENIH PARAMETARA*

Kada uzmemo uzorak(npr 400 knjiga) mozemo izracunati varijansu ocena-to je estimator(s na kvadrat), ali to je samo procena, da bi videli koliko je to pouzdano koristimo INTERVAL POVERENJA. Formula: (n-1) puta s na kvadrat/ prava varijanse. Ukazuje na to koliko smo sigurni da prava  varijansa leži između dve vrednosti. Ako je interval uzak, procena je precizna ali ako je širok ima nesigurnosti u proceni.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

atributi = ['rating']
df = pd.read_csv("books.csv", usecols=atributi)
print(df.head())
print(df.describe())

df.hist(column='rating', density=True, bins=30, color='skyblue', edgecolor='black')
plt.title("Distribucija prosečnih ocena knjiga")
plt.xlabel("Prosečna ocena")
plt.ylabel("Gustina verovatnoće")
plt.show()

estimatori_varijanse = []
velicina_uzorka = 100  
broj_estimacija_varijansi = 50

for i in range(broj_estimacija_varijansi):
    uzorak = df.sample(velicina_uzorka, replace=True)
    estimatori_varijanse.append(uzorak.var()['rating'])

pd.DataFrame(estimatori_varijanse).plot(kind="density", figsize=(8,6), color='purple')
plt.title("Procena gustine verovatnoće varijanse ocena knjiga")
plt.xlabel("Varijansa")
plt.show()

pd.DataFrame(estimatori_varijanse).hist(density=True, bins=15, color='violet', edgecolor='black')
plt.title("Histogram procenjenih varijansi (ocena knjiga)")
plt.xlabel("Varijansa")
plt.ylabel("Frekvencija")
plt.show()

alpha = 0.05  
s2 = estimatori_varijanse[0]

Kvantil1 = stats.chi2.ppf(1 - alpha / 2, df=velicina_uzorka - 1)
Kvantil2 = stats.chi2.ppf(alpha / 2, df=velicina_uzorka - 1)
interval_poverenja = ((velicina_uzorka - 1) * s2 / Kvantil1,
                      (velicina_uzorka - 1) * s2 / Kvantil2)
print(f"\nInterval poverenja (95%) za varijansu prosečne ocene:")
print(interval_poverenja)

intervali_poverenja = []

for i in range(broj_estimacija_varijansi):
    interval = ((velicina_uzorka - 1) * estimatori_varijanse[i] / Kvantil1,
                (velicina_uzorka - 1) * estimatori_varijanse[i] / Kvantil2)
    intervali_poverenja.append(interval)

intervali_poverenja = np.array(intervali_poverenja)


plt.figure(figsize=(9, 8))
plt.errorbar(x=np.arange(0, broj_estimacija_varijansi),
             y=estimatori_varijanse,
             yerr=[estimatori_varijanse - intervali_poverenja[:, 0],
                   intervali_poverenja[:, 1] - estimatori_varijanse],
             fmt='o', ecolor='gray', capsize=4)
plt.hlines(y=df.var()['rating'], xmin=0, xmax=broj_estimacija_varijansi,
           color='red', linewidth=2, label='Varijansa cele populacije')
plt.title("Intervali poverenja za varijansu prosečne ocene knjiga")
plt.xlabel("Broj uzorka")
plt.ylabel("Varijansa")
plt.legend()
plt.show()

U ovoj estimaciji prikazujemo kako racunamo interval poverenja za varijansu parametra- procenjujemo koliko varira prosecna ocena knjiga i koliki je interval poverenja za tu varijansu.
Procenjujemo varijansu prosečnih ocena knjiga pomoću više uzoraka. 
Crvena linija na grafu pokazuje stvarnu varijansu cele baze a tačkice i stubovi su procene iz uzoraka.

*TESTIRANJE PARAMETARSKIH HIPOTEZA*

Ovde ispitujemo oblik raspodele podataka (poreklo podataka), da li poticu iz normalne raspodele. Ovde smo koritili T-test koji proverava da li se prosecna ocena drasticno razlikuje od neke poznate vrednosti (H0 nema razlike, H1 postoji razlika). Ako imamo veliko odstupanje i malo širenje podataka-->razlika je znacajna. nakon sto izracunamo t i dobijemo p vrednost: ako je p manje od alpha ODBACUJEMO H0 (postoji razlika), ali ako je p vece od alpha NE ODBACUJEMO H0 (nema znacajne razlike)

In [ ]:
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
df = pd.read_csv("books.csv", nrows=200)

data = df["rating"]

h0 = 3  
alpha = 0.05  
n = len(data)
mean_sample = np.mean(data)
std_sample = np.std(data, ddof=1)

t_stat, p_value = stats.ttest_1samp(data, h0)

interval_poverenja = stats.t.interval(   #formula za interval poverenja 95% za sredinu
    1 - alpha,                           #--> stvarna sredina se s verovatnoćom od 95% nalazi u tom intervalu
    df=n - 1,
    loc=mean_sample,
    scale=std_sample / np.sqrt(n)
)

print(f"Broj podataka: {n}")
print(f"Aritmetička sredina uzorka: {mean_sample:.3f}")
print(f"Standardna devijacija: {std_sample:.3f}")
print(f"t-statistika: {t_stat:.3f}")
print(f"p-vrednost: {p_value:.5f}")
print(f"95% interval poverenja: {interval_poverenja}")

if p_value < alpha:
    print("\nOdbacujemo H₀: Postoji statistički značajna razlika u odnosu na 3.")
else:
    print("\nNe odbacujemo H₀: Nema značajne razlike u odnosu na 3.")

plt.figure(figsize=(8,5))
sns.histplot(data, kde=True, bins=20, color="skyblue")
plt.axvline(mean_sample, color='red', linestyle='--', label=f"Mean = {mean_sample:.2f}")
plt.axvline(h0, color='green', linestyle='--', label="H₀ = 3")
plt.title("Distribucija uzorka i poređenje sa H₀")
plt.legend()
plt.show()

U završnom koraku testiramo hipotezu, uzorak je iz normalne raspodele i ne znamo populacionu standardnu devijaciju pa koristimo t-test.
Za prosecnu vrednost h0 smo stavili 3 a za h1 da prosecna vrednost ne sme da bude 3